# Part 5 sandbox: the host does not trust the container

A literate walkthrough of `minimal_loop/docker_session.py`, written as a notebook so you can read the idea, run it, and see the real output next to it.

**The analogy.** The container is a contractor working in a locked room who slides notes under the door. The host is the site manager. The manager reads every note, but before acting on one the manager checks: is it the right size, is it written on the right form, are the numbers sane? A contractor who hands over a 40 GB note, or a form with `NaN` in the measurement box, gets the room locked and the job marked *REVIEW*. Nobody guesses what they meant.

**What the review found, and what each fix does**

| # | Finding | Fix in `docker_session.py` |
|---|---|---|
| 1 | The submission came from the sandbox unchecked (`NaN`, 1e9, a bool) | `validate_reply` checks type, finiteness, range 0 to 100, note length; JSON `NaN`/`Infinity` is rejected at parse time |
| 2 | No size cap, so one endless line could fill host memory | bounded `readline(MAX_LINE + 1)`; over the cap kills the container and the rest is drained and discarded |
| 3 | Image URLs were passed through (`http://...` would be fetched by whatever renders them) | only `data:image/png;base64,` URLs, length cap, must decode, must start with the PNG signature and an `IHDR` chunk |

In [1]:
import sys, json, base64, zlib, struct
sys.path.insert(0, "../minimal_loop")   # this notebook lives in literate/, the code in minimal_loop/
import numpy as np
from docker_session import validate_reply, ProtocolError, _no_constant, MAX_LINE, MAX_STDOUT, MAX_URL, DATA_PREFIX
print("size caps:", dict(MAX_LINE=MAX_LINE, MAX_STDOUT=MAX_STDOUT, MAX_URL=MAX_URL))

size caps: {'MAX_LINE': 6000000, 'MAX_STDOUT': 20000, 'MAX_URL': 4000000}


## 1. The check on its own (no Docker needed)

`validate_reply(reply, expected_id)` either returns a cleaned copy of the reply or raises `ProtocolError`. First a tiny real PNG and an honest reply.

In [2]:
def tiny_png():
    def chunk(t, d): return struct.pack(">I", len(d)) + t + d + struct.pack(">I", zlib.crc32(t + d))
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", 1, 1, 8, 0, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(b"\x00\x80")) + chunk(b"IEND", b""))
PNG_URL = DATA_PREFIX + base64.b64encode(tiny_png()).decode()

honest = {"id": 2, "stdout": "[submitted]", "error": "", "timed_out": False, "variables": {"x": "float"},
          "submission": {"void_ratio_pct": 0.4, "note": "ok"}, "images": [["mask", PNG_URL]]}
clean = validate_reply(honest, 2)
print("accepted:", {k: (v if k != "images" else f"{len(v)} image(s)") for k, v in clean.items()})

accepted: {'stdout': '[submitted]', 'error': '', 'timed_out': False, 'variables': {'x': 'float'}, 'submission': {'void_ratio_pct': 0.4, 'note': 'ok'}, 'images': '1 image(s)'}


## 2. Lies the host must refuse

Each row is one forged reply. `expect` is always a refusal. The loop turns any refusal into *REVIEW*, never into PASS.

In [3]:
def lie(**over):
    r = json.loads(json.dumps(honest)); r.update(over); return r

cases = {
  "NaN measurement":        lambda: validate_reply(json.loads(json.dumps(honest).replace("0.4", "NaN"), parse_constant=_no_constant), 2),
  "1e9 measurement":        lambda: validate_reply(lie(submission={"void_ratio_pct": 1e9, "note": ""}), 2),
  "negative measurement":   lambda: validate_reply(lie(submission={"void_ratio_pct": -5, "note": ""}), 2),
  "bool measurement":       lambda: validate_reply(lie(submission={"void_ratio_pct": True, "note": ""}), 2),
  "extra submission field": lambda: validate_reply(lie(submission={"void_ratio_pct": 1, "note": "", "verdict": "PASS"}), 2),
  "300+ char note":         lambda: validate_reply(lie(submission={"void_ratio_pct": 1, "note": "x" * 301}), 2),
  "wrong id":               lambda: validate_reply(lie(id=99), 2),
  "http image URL":         lambda: validate_reply(lie(images=[["t", "http://169.254.169.254/latest/meta-data"]]), 2),
  "data URL, not a PNG":    lambda: validate_reply(lie(images=[["t", DATA_PREFIX + base64.b64encode(b"hello").decode()]]), 2),
  "three images":           lambda: validate_reply(lie(images=[["t", PNG_URL]] * 3), 2),
  "huge stdout":            lambda: validate_reply(lie(stdout="x" * 30_000), 2),
}
for name, f in cases.items():
    try:
        f(); print(f"ACCEPTED (bad!)  {name}")
    except ProtocolError as e:
        print(f"refused          {name:24s} -> {e}")

refused          NaN measurement          -> non-finite JSON constant NaN
refused          1e9 measurement          -> submission void_ratio_pct
refused          negative measurement     -> submission void_ratio_pct
refused          bool measurement         -> submission void_ratio_pct
refused          extra submission field   -> submission shape
refused          300+ char note           -> submission note
refused          wrong id                 -> wrong id
refused          http image URL           -> image url
refused          data URL, not a PNG      -> image is not a PNG
refused          three images             -> images
refused          huge stdout              -> stdout


### A check that must *not* refuse

A validator that refuses honest replies is its own bug. The first version of this check capped `variables` at 60 entries and rejected anything larger. Replaying the 634 recorded cells (notebook 05d) found 8 honest cells, from two runs, that the check killed, because an honest model defined more than 60 variables. `variables` is informational (the loop shows the first 25), so the host now **clips** it instead of refusing. Wrong *types* are still refused.

In [4]:
many = lie(variables={f"v{i}": "int = 1" for i in range(200)})
out = validate_reply(many, 2)
print("accepted, clipped to", len(out["variables"]), "of 200 variables")
try:
    validate_reply(lie(variables={"x": 5}), 2)
except ProtocolError as e:
    print("a non-string variable description is still refused ->", e)

accepted, clipped to 60 of 200 variables
a non-string variable description is still refused -> variables entries


## 3. The same lies from inside a real container

Now the model's code runs in the Docker sandbox described in `docker_session.py` (no network, read-only filesystem, all capabilities dropped, 1 GB memory). The cell writes its own forged reply straight onto the protocol channel (file descriptor 3, the private copy of stdout that the runner keeps). The host must end the episode as REVIEW.

`run_episode4` is the real loop; the model is a scripted stand-in that emits the cell we give it.

In [5]:
from docker_session import DockerSession
from loop4 import run_episode4

IMG = np.full((16, 16), 200, np.uint8)
def scripted(*cells):
    it = iter(cells)
    return lambda m: (f"Purpose: scripted.\n```python\n{next(it, 'print(1)')}\n```", {"prompt": 10, "completion": 5})
def go(*cells):
    return run_episode4(IMG, "q", scripted(*cells), session_cls=DockerSession, cell_timeout=3)
def forge(payload):   # id 2 = the id the host expects for the first model cell
    return f"import os\nos.write(3, ({payload!r}).encode() + b'\\n')"
GOOD = '{"id": 2, "stdout": "", "error": "", "timed_out": false, "variables": {}, "images": [], "submission": %s}'

attacks = {
  "honest submit(0.4)":        "submit(0.4)",
  "forged NaN submission":     forge(GOOD % '{"void_ratio_pct": NaN, "note": "x"}'),
  "forged 1e9 submission":     forge(GOOD % '{"void_ratio_pct": 1e9, "note": "x"}'),
  "forged http image URL":     forge((GOOD % 'null').replace('"images": []', '"images": [["t", "http://169.254.169.254/x"]]')),
}
for name, cell in attacks.items():
    ep = go(cell, "submit(0.1)")
    print(f"{name:26s} -> {ep.verdict.status:6s} {ep.verdict.reason[:95]}")

honest submit(0.4)         -> PASS   rule: 0.40 <= 1.50


forged NaN submission      -> REVIEW sandbox_protocol_error: answer was not the expected reply to this cell (non-finite JSON constan


forged 1e9 submission      -> REVIEW sandbox_protocol_error: answer was not the expected reply to this cell (submission void_ratio_p


forged http image URL      -> REVIEW sandbox_protocol_error: answer was not the expected reply to this cell (image url)


## 4. The endless line

The nastiest case for the host is not a wrong value but a reply that never ends. The cell below writes 1 MB chunks forever without a newline. The bounded read stops at `MAX_LINE`, the container is killed, and the pump keeps draining the pipe so the container can never be left blocked on a full pipe while the kill happens.

*Why the draining matters (found while writing these tests):* the first version stopped reading at the cap. The container then blocked on a full pipe, `docker kill` hung, and the Docker daemon stopped answering for several minutes until the client processes were killed. The fix is the drain loop plus `kill()` also killing the local `docker` client and giving `docker kill` a 20 s limit.

In [6]:
import time
t0 = time.time()
ep = go("import os\nwhile True: os.write(3, b'x' * 1000000)", "submit(0.1)")
print(f"{ep.verdict.status}: {ep.verdict.reason}   ({time.time() - t0:.1f}s, steps={ep.steps})")

REVIEW: sandbox_protocol_error: reply longer than the size cap   (0.5s, steps=1)


## 5. Run everything

```bash
cd notebooks/spatialclaw-inspection
.venv/bin/python -I minimal_loop/test_paths5.py        # 16 paths, about 90 s, needs the sc-inspect-sandbox image
```

The 9 original paths (limit, persistence, errors, loops, killed container, memory bomb, forged id, `exit()`) plus 7 new ones (forged NaN / 1e9 / bool / negative submission, forged `http` image URL, forged non-PNG data URL, endless line).

**Not covered:** the sandbox's own isolation flags are tested elsewhere (Part 5 post); a model that finds a kernel or Docker escape is out of scope for these tests; the model here is a scripted stand-in, so no real LLM behaviour is tested.